# Gao development: matched-seed architecture comparison

This experiment compares the unweighted PCBert-Kla baseline and token-gated model on the locked Gao training/validation partition. Five prespecified seeds are paired across architectures. A deterministic logistic-regression control uses only the 27 physicochemical features. Checkpoint selection uses Gao validation AUPRC; the displayed metrics are on that same validation partition and therefore are development results, not an independent performance estimate. Gao internal test and rice are not evaluated here.

## 1. Repository, environment, and data lock

The private repository is cloned using the Kaggle `GITHUB_TOKEN` secret. The token is passed through a temporary Git askpass script and is not printed or embedded in the clone URL.

In [ ]:
import hashlib
import importlib.metadata
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

WORKING_DIR = Path('/kaggle/working')
REPO_DIR = WORKING_DIR / 'MM-Kla'
PROJECT_DIR = REPO_DIR / 'experiment' / 'pcbert_kla_clean'
OUTPUT_ROOT = PROJECT_DIR / 'outputs' / 'kaggle_multiseed_development'
REPO_URL = 'https://github.com/kaleabnega/MM-Kla.git'
if not WORKING_DIR.is_dir():
    raise RuntimeError('A Kaggle runtime is required.')
if not REPO_DIR.exists():
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    if not token:
        raise RuntimeError('Kaggle secret GITHUB_TOKEN is missing.')
    with tempfile.TemporaryDirectory() as temporary:
        askpass = Path(temporary) / 'askpass.sh'
        askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) printf "%s\n" x-access-token ;;\n  *Password*) printf "%s\n" "$MM_KLA_GITHUB_TOKEN" ;;\n  *) exit 1 ;;\nesac\n')
        askpass.chmod(0o700)
        env = os.environ.copy()
        env.update({'GIT_ASKPASS': str(askpass), 'GIT_TERMINAL_PROMPT': '0', 'MM_KLA_GITHUB_TOKEN': token})
        try:
            clone = subprocess.run(['git', '-c', 'credential.helper=', 'clone', '--depth', '1', '--branch', 'main', REPO_URL, str(REPO_DIR)], env=env, capture_output=True, text=True, check=False)
        finally:
            env.pop('MM_KLA_GITHUB_TOKEN', None)
            del token
    if clone.returncode:
        raise RuntimeError('Private clone failed; check Internet and secret access.')
elif not (REPO_DIR / '.git').is_dir():
    raise RuntimeError(f'{REPO_DIR} is not a Git checkout.')
if not (PROJECT_DIR / 'src/pcbert_kla_clean/simple_baselines.py').is_file():
    raise RuntimeError('The current experiment code is missing; start a fresh Kaggle session or update the checkout.')
COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, capture_output=True, text=True, check=True).stdout.strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'], cwd=PROJECT_DIR, check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a GPU in Kaggle settings.')
LOCK_PATH = PROJECT_DIR / 'review_data/locked_source_study_v1/protocol_lock.json'
LOCK_SHA256 = hashlib.sha256(LOCK_PATH.read_bytes()).hexdigest()
subprocess.run([sys.executable, 'scripts/run_replication.py', '--run', 'source-study-check'], cwd=PROJECT_DIR, check=True)
runtime = {'git_commit': COMMIT, 'source_lock_sha256': LOCK_SHA256, 'python': sys.version.split()[0], 'gpu': torch.cuda.get_device_name(0), 'packages': {name: importlib.metadata.version(name) for name in ('numpy', 'pandas', 'scikit-learn', 'torch', 'transformers')}}
(WORKING_DIR / 'mm_kla_multiseed_runtime.json').write_text(json.dumps(runtime, indent=2, sort_keys=True) + '\n')
print(json.dumps(runtime, indent=2, sort_keys=True))

## 2. Matched neural-model runs

Both architectures use ProtBert (`Rostlab/prot_bert`), four trainable encoder layers, 30 epochs maximum, batch size 4, SGD at 0.003, no positive weighting, and a fixed 0.5 classification threshold. The runner selects the checkpoint with highest Gao validation AUPRC. The same seed list is used for both architectures. No model checkpoints are saved to limit Kaggle disk use; validation predictions and run configurations are retained.

In [ ]:
SEEDS = (42, 123, 2025, 3407, 777)
ARCHITECTURES = ('baseline', 'token_gated')
RUN_DIRS = {}
RUN_ROOT = OUTPUT_ROOT / 'source_study' / 'runs'

def matching_runs(architecture, seed):
    matches = []
    for config_path in RUN_ROOT.glob('*/run_config.json'):
        config = json.loads(config_path.read_text())
        if (config.get('run') == 'source-study-development'
                and config.get('architecture') == architecture
                and config.get('model_seed') == seed
                and config.get('positive_weighting') == 'none'
                and config.get('model_name') == 'Rostlab/prot_bert'
                and config.get('encoder_layers') == 4
                and config.get('epochs') == 30
                and config.get('batch_size') == 4
                and config.get('optimizer') == 'sgd'
                and config.get('learning_rate') == 0.003
                and config.get('weight_decay') == 0.0
                and config.get('scheduler') == 'none'
                and config.get('monitor_metric') == 'AUPRC'
                and config.get('decision_threshold') == 0.5
                and config.get('source_lock_sha256') == LOCK_SHA256
                and config.get('training_rows') == 3599
                and config.get('validation_rows') == 519
                and config.get('test_rows') == 0
                and not config.get('extra_test_rows')
                and (config_path.parent / 'validation_metrics.json').is_file()
                and (config_path.parent / 'validation_predictions.csv').is_file()):
            matches.append(config_path.parent)
    return matches

for architecture in ARCHITECTURES:
    for seed in SEEDS:
        matches = matching_runs(architecture, seed)
        if not matches:
            command = [sys.executable, 'scripts/run_replication.py',
                       '--run', 'source-study-development', '--architecture', architecture,
                       '--positive-weighting', 'none', '--model-name', 'Rostlab/prot_bert',
                       '--encoder-layers', '4', '--epochs', '30', '--batch-size', '4',
                       '--device', 'cuda:0', '--optimizer', 'sgd', '--learning-rate', '0.003',
                       '--weight-decay', '0', '--scheduler', 'none', '--seed', str(seed),
                       '--output-dir', str(OUTPUT_ROOT)]
            print('Running:', architecture, seed, flush=True)
            subprocess.run(command, cwd=PROJECT_DIR, check=True)
            matches = matching_runs(architecture, seed)
        if len(matches) != 1:
            raise RuntimeError(f'Run not uniquely identified: {architecture}, {seed}: {matches}')
        RUN_DIRS[(architecture, seed)] = matches[0]
        print('Saved run:', matches[0], flush=True)

## 3. Physicochemical-feature control

A logistic regression classifier is fitted once on Gao training rows using the same 27 descriptors. Its MinMax scaler is fitted on training rows only. This control is deterministic and has no training seed; it does **not** match the information available to ProtBert-based models because it excludes sequence embeddings. It is an initial feature-only control, not a substitute for a matched mean-pooled-PLM-plus-features baseline.

In [ ]:
sys.path.insert(0, str(PROJECT_DIR / 'src'))
from pcbert_kla_clean.experiments import save_predictions
from pcbert_kla_clean.simple_baselines import feature_logistic_validation
from pcbert_kla_clean.source_study import load_locked_partitions

gao_path = PROJECT_DIR / 'review_data/gao_2020_source_split/source_split_manifest.csv'
rice_path = PROJECT_DIR / 'review_data/source_study_audit_2026_11_01/rice_homology_screened_candidates.csv'
development, validation_indices, _, _ = load_locked_partitions(gao_path, rice_path, LOCK_PATH)
scores, feature_metrics = feature_logistic_validation(development, validation_indices)
feature_dir = OUTPUT_ROOT / 'feature_logistic'
feature_dir.mkdir(parents=True, exist_ok=True)
(feature_dir / 'validation_metrics.json').write_text(json.dumps(feature_metrics, indent=2, sort_keys=True) + '\n')
(feature_dir / 'run_config.json').write_text(json.dumps({'model': 'logistic_regression_27_features', 'git_commit': COMMIT, 'source_lock_sha256': LOCK_SHA256, 'training_rows': len(development.records) - len(validation_indices), 'validation_rows': len(validation_indices), 'test_rows': 0, 'scaler': 'MinMaxScaler_train_only', 'C': 1.0, 'solver': 'lbfgs', 'max_iter': 1000, 'class_weight': None, 'decision_threshold': 0.5}, indent=2, sort_keys=True) + '\n')
save_predictions(feature_dir / 'validation_predictions.csv', development, validation_indices, scores, 0.5)
print(json.dumps(feature_metrics, indent=2, sort_keys=True))
print('Saved feature-only control:', feature_dir)

## 4. Validation summary

Per-seed results, paired token-gated-minus-baseline differences, and mean/sample standard deviation across the five neural-model seeds are computed from saved metrics. The feature-only control is reported once because it is deterministic. Mean and standard deviation across training seeds do not measure uncertainty from the Gao partition or establish external generalization.

In [ ]:
import pandas as pd
from IPython.display import display

METRICS = ('AUPRC', 'AUC', 'MCC', 'F1', 'Pre', 'Rec', 'SP', 'ACC')
rows = []
for (architecture, seed), run_dir in RUN_DIRS.items():
    config = json.loads((run_dir / 'run_config.json').read_text())
    values = json.loads((run_dir / 'validation_metrics.json').read_text())
    rows.append({'architecture': architecture, 'seed': seed, 'selected_epoch': config['selected_epoch'], **{metric: values[metric] for metric in METRICS}})
per_seed = pd.DataFrame(rows).sort_values(['architecture', 'seed']).reset_index(drop=True)
display(per_seed.round(4))
comparison = per_seed.pivot(index='seed', columns='architecture', values=list(METRICS))
paired = pd.DataFrame({metric: comparison[(metric, 'token_gated')] - comparison[(metric, 'baseline')] for metric in METRICS})
display(paired.round(4).rename_axis('seed'))
summary = per_seed.groupby('architecture')[list(METRICS)].agg(['mean', 'std'])
display(summary.round(4))
display(pd.DataFrame([{'architecture': 'feature_logistic_27_features', 'seed': 'deterministic', **feature_metrics}]).round(4))
per_seed.to_csv(WORKING_DIR / 'mm_kla_multiseed_per_seed.csv', index=False)
paired.to_csv(WORKING_DIR / 'mm_kla_multiseed_paired_differences.csv')
summary.to_csv(WORKING_DIR / 'mm_kla_multiseed_summary.csv')
print('Saved validation summaries to', WORKING_DIR)
print('Saved validation predictions to', OUTPUT_ROOT)